<a href="https://colab.research.google.com/github/DrDourado/ME_640_Optimum_Design_Fall_2026/blob/main/ME_640_12_End_to_End_Optimal_Design_Framework_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ME 640 — End-to-End Optimal Design Framework: Solution
## Four benchmarks from problem definition to fuzzy RBDO

For **each** benchmark we have:

$$
\boxed{\text{define}\to\text{deterministic}\to\text{robust}\to\text{RBDO}\to\text{fuzzy RBDO}\to\text{verify}}
$$

### Benchmarks
1. Tension/compression spring.
2. Welded beam.
3. Golinski speed reducer.
4. Cantilever-beam RBDO benchmark.

The distinction is central:

- **Deterministic:** optimize at nominal conditions.
- **Robust:** reduce adverse average behavior and/or variability.
- **RBDO:** explicitly enforce a target probability of failure or reliability index.
- **Fuzzy RBDO:** enforce safety under fuzzy/possibilistic uncertainty; no probability interpretation is implied.

### References
- Belegundu, A.D. (1985), [A Study of Mathematical Programming Methods for Structural Optimization.](https://doi.org/10.1002/nme.1620210905)
- Coello, C.A. (2000), [Use of a self-adaptive penalty approach for engineering optimization problems](https://doi.org/10.1016/S0166-3615(99)00046-9)
- Golinski, J. (1973), [An adaptive optimization system applied to machine synthesis](https://doi.org/10.1016/0094-114X(73)90018-9)
- Dourado, A.D, et al. (2019), [Fuzzy Reliability-Based Optimization for Engineering System Design](https://doi.org/10.1007/s40815-019-00655-5)

**Important:** probabilistic and fuzzy uncertainty models added to the spring, welded beam, and speed reducer are instructional extensions of their deterministic benchmark formulations.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from scipy.stats import norm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
np.random.seed(640)
plt.rcParams["figure.figsize"]=(7.5,5)

def solve(obj,cons,bounds,seed=1,it=120,rho=1e7):
    F=lambda x: obj(x)+rho*np.sum(np.maximum(np.asarray(cons(x),float),0)**2)
    return differential_evolution(F,bounds,seed=seed,maxiter=it,popsize=9,polish=True,tol=1e-7)

def beta_pf(p,n):
    return -norm.ppf(np.clip(p,.5/n,1-.5/n))

def acut(tri,a):
    L,N,R=tri
    return L+a*(N-L),R-a*(R-N)

def check(x,cons):
    g=np.asarray(cons(x))
    return pd.DataFrame({"j":range(1,len(g)+1),"g":g,"feasible":g<=1e-6})

# 1. Concepts to use throughout

### Deterministic
$$
\min f(\mathbf d)\quad\text{s.t.}\quad g_j(\mathbf d)\le0.
$$

### Mean–variance robustness
$$
J_R=\mathbb E[f]+\lambda\sigma_f.
$$

### Probabilistic reliability
With safety margin \(G=-g\),
$$
P_f=P(G<0),\qquad\beta=-\Phi^{-1}(P_f).
$$

### Fuzzy uncertainty
$$
\tilde x=(x_L,x_N,x_R),
$$
$$
X_\alpha=[x_L+\alpha(x_N-x_L),\,x_R-\alpha(x_R-x_N)].
$$

# 2. Benchmark I — Tension/Compression Spring

### Engineering description
A helical compression spring must carry an axial load while using as little material as possible. The designer selects the **wire diameter** $d$, **mean coil diameter** $D$, and **number of active coils** $N$. A lighter/smaller spring is desirable, but reducing these dimensions increases shear stress and deflection and can violate geometric limits. This makes the spring a compact nonlinear constrained-design benchmark.

### Mathematical formulation
The design vector and material/volume proxy are
$$
\mathbf d=[d,D,N],\qquad \min f(\mathbf d)=(N+2)Dd^2.
$$
The standard normalized constraints are
$$
g_1=1-\frac{D^3N}{71785d^4}\le0 \qquad \text{(deflection)},
$$
$$
g_2=\frac{4D^2-dD}{12566(Dd^3-d^4)}+\frac{1}{5108d^2}-1\le0
\qquad \text{(shear stress)},
$$
$$
g_3=1-\frac{140.45d}{D^2N}\le0 \qquad \text{(frequency/surge-related constraint)},
$$
$$
g_4=\frac{d+D}{1.5}-1\le0 \qquad \text{(outside-diameter/geometric limit)}.
$$
Bounds are
$$
0.05\le d\le1,\qquad0.25\le D\le1.3,\qquad2\le N\le15.
$$

### Uncertainty model used in this notebook
To extend the deterministic benchmark without changing its basic structure, multiplicative demand factors $q_1,q_2,q_3$ are introduced into the first three mechanical constraints. The nominal benchmark corresponds to $q_k=1$. For probabilistic RBDO,
$$
q_k\sim\mathcal N(1,0.05^2),
$$
and for the fuzzy formulation,
$$
\tilde q_k=(0.90,1.00,1.10).
$$
These uncertainty definitions are **instructional extensions**, not part of the original deterministic benchmark.

## 2.1 — Model implementation

Encode the stated objective, constraints/limit states, and design bounds before introducing uncertainty.

In [ ]:
sb=[(.05,1),(.25,1.3),(2,15)]
def sf(x): d,D,N=x; return (N+2)*D*d*d
def sg(x,q=np.ones(3)):
 d,D,N=x; q1,q2,q3=q
 return np.array([1-q1*D**3*N/(71785*d**4),
 q2*((4*D**2-d*D)/(12566*(D*d**3-d**4))+1/(5108*d**2))-1,
 1-(140.45/q3)*d/(D**2*N),(d+D)/1.5-1])

## 2.2 — Deterministic optimization

All uncertain quantities are fixed at their nominal values:

$$
\min_{\mathbf d} f(\mathbf d)
\quad\mathrm{s.t.}\quad
g_j(\mathbf d)\le0.
$$

The resulting design is the nominal baseline. The original constraints are evaluated after optimization to verify feasibility independently of the optimizer's status.

In [ ]:
sd=solve(sf,lambda x:sg(x),sb,1,150); print("DET",sd.x,sf(sd.x)); display(check(sd.x,lambda x:sg(x)))

DET [0.05309828 0.39156829 9.50486133] 0.01270134949482602


,j,g,feasible
0,1,-0.000029,True
1,2,-0.000017,True
2,3,-4.117306,True
3,4,-0.703556,True


## 2.3 — Uncertainty quantification of the deterministic optimum

Before redesigning the system, propagate the specified uncertainty through the deterministic optimum. For each safety margin $G_j=-g_j$,

$$
P_{f,j}\approx \frac{N(G_j<0)}{N},
\qquad
\beta_j=-\Phi^{-1}(P_{f,j}).
$$

This reveals whether the nominally feasible optimum remains reliable when uncertainty is introduced.

In [ ]:
rng=np.random.default_rng(11); Q=1+.05*rng.standard_normal((3000,3))
def srel(x,Q=Q):
 G=np.array([-sg(x,q)[:3] for q in Q]); pf=(G<0).mean(0)
 return pf,np.array([beta_pf(p,len(Q)) for p in pf])

## 2.4 — Robust optimization

**Purpose:** reduce sensitivity to uncertainty rather than enforce a prescribed failure probability.

Use the maximum uncertain mechanical-constraint demand as a scalar response. Penalize its positive mean and its standard deviation, while retaining nominal geometric feasibility.

A generic form is

$$
J_R(\mathbf d)=f(\mathbf d)+\lambda\,\sigma_D(\mathbf d)
+
ρ\,[\max(\mu_D(\mathbf d),0)]^2,
$$

where $D$ is an uncertainty-dependent demand/violation measure. The mean term discourages adverse average behavior and the standard-deviation term discourages variability.

In [ ]:
def srob(x):
 vals=np.array([sg(x,q)[:3] for q in Q[:600]]); z=np.max(vals,1)
 return sf(x)+2*np.std(z)+1e5*max(z.mean(),0)**2+1e7*np.sum(np.maximum(sg(x)[3:],0)**2)
sr=differential_evolution(srob,sb,seed=2,maxiter=65,popsize=8,polish=True)

## 2.5 — Probabilistic RBDO

For a specified: $\beta_T=2.5$, the design must satisfy
$$
\beta_j(\mathbf d)\ge\beta_T
$$

for every probabilistic mechanical limit state. The outer optimizer therefore searches for the least-cost design that satisfies an explicit reliability requirement.

This differs fundamentally from the robust stage: here the acceptance criterion has a probability-based interpretation.

In [ ]:
def srb(x):
 b=srel(x,Q[:1600])[1]
 return sf(x)+1e5*np.sum(np.maximum(2.5-b,0)**2)+1e7*np.sum(np.maximum(sg(x)[3:],0)**2)
sp=differential_evolution(srb,sb,seed=3,maxiter=70,popsize=8,polish=True)

## 2.6 — Fuzzy RBDO

At alpha=0 use the upper endpoints of the fuzzy demand factors because larger q increases mechanical demand.

For $\widetilde x=(x_L,x_N,x_R)$,
$$
X_\alpha=[x_L+\alpha(x_N-x_L),\;x_R-\alpha(x_R-x_N)].
$$

The implementation uses **full-support protection** at $\alpha=0$. The adverse endpoint combination is propagated through the benchmark constraints and the design is optimized subject to feasibility at that realization.

This is a possibilistic/imprecision-based requirement and is not equivalent to a target $P_f$ or $\beta$.

In [ ]:
def sfg(x,a=0):
 q=np.array([acut((.9,1,1.1),a)[1]]*3); return sg(x,q)
sfu=solve(sf,lambda x:sfg(x,0),sb,4,80)
display(pd.DataFrame([[n,*z,sf(z),*srel(z)[1]] for n,z in
 [("Deterministic",sd.x),("Robust",sr.x),("RBDO",sp.x),("Fuzzy RBDO",sfu.x)]],
 columns=["Method","d","D","N","Objective","beta1","beta2","beta3"]))

,Method,d,D,N,Objective,beta1,beta2,beta3
0,Deterministic,0.053098,0.391568,9.504861,0.012701,0.018383,-0.045971,3.587915
1,Robust,0.056069,0.461117,7.503893,0.013777,0.733369,0.363810,3.587915
2,RBDO,0.056081,0.409505,11.942665,0.017957,2.681316,2.512144,3.587915
3,Fuzzy RBDO,0.055772,0.412632,8.990457,0.014106,-1.965699,1.989313,3.587915


## 2.7 — Benchmark comparison and interpretation

Compare the four resulting designs in terms of:

1. objective value and percentage increase from the deterministic baseline;
2. active/governing constraints;
3. reliability of the robust design under the probabilistic model;
4. objective penalty required to reach the RBDO target;
5. additional protection introduced by the full-support fuzzy formulation.


# 3. Benchmark II — Welded Beam

### Engineering description
A cantilevered beam is welded to a rigid support and subjected to an end load $P$. The designer chooses the **weld size** $h$, **weld length** $l$, **beam depth** $t$, and **beam thickness** $b$. The objective approximates fabrication/material cost. Making the structure smaller reduces cost but increases weld shear stress, beam bending stress, tip deflection, and susceptibility to buckling.

### Mathematical formulation
$$
\mathbf d=[h,l,t,b],
\qquad
\min f=1.10471h^2l+0.04811tb(14+l).
$$
With $L=14$, define
$$
M=P\left(L+\frac l2\right),\quad
R=\sqrt{\frac{l^2}{4}+\left(\frac{h+t}{2}\right)^2},
$$
$$
J=2\sqrt2hl\left[\frac{l^2}{12}+\left(\frac{h+t}{2}\right)^2\right],
$$
$$
\tau_1=\frac{P}{\sqrt2hl},\qquad
\tau_2=\frac{MR}{J},
$$
$$
\tau=\sqrt{\tau_1^2+2\tau_1\tau_2\frac{l}{2R}+\tau_2^2}.
$$
The beam normal stress and tip deflection are
$$
\sigma=\frac{6PL}{bt^2},\qquad
\delta=\frac{4PL^3}{Et^3b}.
$$
The buckling load is
$$
P_c=\frac{4.013E}{L^2}\sqrt{\frac{t^2b^6}{36}}
\left(1-\frac{t}{2L}\sqrt{\frac{E}{4G}}\right).
$$
The normalized constraints implemented in the notebook are
$$
g_1=\frac{\tau}{13600}-1\le0,\quad
 g_2=\frac{\sigma}{30000}-1\le0,
$$
$$
g_3=\frac hb-1\le0,\quad
 g_4=\frac{\delta}{0.25}-1\le0,\quad
 g_5=\frac{P}{P_c}-1\le0.
$$
The variable bounds are
$$
0.125\le h\le2,\;0.1\le l\le10,\;0.1\le t\le10,\;0.125\le b\le2.
$$
Nominal properties are $P=6000$, $E=30\times10^6$, and $G=12\times10^6$.

### Uncertainty model used in this notebook
For probabilistic analysis, $P$ has COV 8% and $E,G$ have COV 4%. The fuzzy extension uses $P\pm12\%$ and $E,G\pm8\%$ triangular supports about their nominal values. These are instructional uncertainty assumptions.

## 3.1 — Model implementation

Encode the stated objective, constraints/limit states, and design bounds before introducing uncertainty.

In [ ]:
wb=[(.125,2),(.1,10),(.1,10),(.125,2)]
def wf(x):
 h,l,t,b=x; return 1.10471*h*h*l+.04811*t*b*(14+l)
def wg(x,p=(6000.,30e6,12e6)):
 h,l,t,b=x; P,E,G=p; L=14.; M=P*(L+l/2); R=np.sqrt(l*l/4+((h+t)/2)**2)
 J=2*np.sqrt(2)*h*l*(l*l/12+((h+t)/2)**2); t1=P/(np.sqrt(2)*h*l); t2=M*R/J
 tau=np.sqrt(t1*t1+2*t1*t2*l/(2*R)+t2*t2); sig=6*P*L/(b*t*t)
 delta=4*P*L**3/(E*t**3*b)
 Pc=(4.013*E*np.sqrt(t*t*b**6/36)/L**2)*(1-t/(2*L)*np.sqrt(E/(4*G)))
 return np.array([tau/13600-1,sig/30000-1,h/b-1,delta/.25-1,P/Pc-1])

## 3.2 — Deterministic optimization

All uncertain quantities are fixed at their nominal values:

$$
\min_{\mathbf d} f(\mathbf d)
\quad\mathrm{s.t.}\quad
g_j(\mathbf d)\le0.
$$

The resulting design is the nominal baseline. The original constraints are evaluated after optimization to verify feasibility independently of the optimizer's status.

In [ ]:
wd=solve(wf,lambda x:wg(x),wb,5,150); print("DET",wd.x,wf(wd.x)); display(check(wd.x,lambda x:wg(x)))

DET [0.20572878 3.47053669 9.03666821 0.20572942] 1.7248634557541265


,j,g,feasible
0,1,-1.072486e-05,True
1,2,-8.715482e-06,True
2,3,-3.073675e-06,True
3,4,-9.421621e-01,True
4,5,4.581140e-08,True


## 3.3 — Uncertainty quantification of the deterministic optimum

Before redesigning the system, propagate the specified uncertainty through the deterministic optimum. For each safety margin $G_j=-g_j$,

$$
P_{f,j}\approx \frac{N(G_j<0)}{N},
\qquad
\beta_j=-\Phi^{-1}(P_{f,j}).
$$

This reveals whether the nominally feasible optimum remains reliable when uncertainty is introduced.

In [ ]:
rng=np.random.default_rng(22); n=2500
WP=np.c_[rng.normal(6000,480,n),rng.normal(30e6,1.2e6,n),rng.normal(12e6,.48e6,n)]
def wrel(x,PP=WP):
 G=np.array([-wg(x,p) for p in PP]); pf=(G<0).mean(0)
 return pf,np.array([beta_pf(p,len(PP)) for p in pf])

## 3.4 — Robust optimization

**Purpose:** reduce sensitivity to uncertainty rather than enforce a prescribed failure probability.

Use the maximum normalized constraint demand across shear, stress, geometry, deflection, and buckling. Penalize both its positive mean and standard deviation under uncertain P, E, and G.

A generic form is

$$
J_R(\mathbf d)=f(\mathbf d)+\lambda\,\sigma_D(\mathbf d)
+
\rho\,[\max(\mu_D(\mathbf d),0)]^2,
$$

where \(D\) is an uncertainty-dependent demand/violation measure. The mean term discourages adverse average behavior and the standard-deviation term discourages variability.

In [ ]:
def wrob(x):
 v=np.array([np.max(wg(x,p)) for p in WP[:500]])
 return wf(x)+2*np.std(v)+1e5*max(v.mean(),0)**2
wr=differential_evolution(wrob,wb,seed=7,maxiter=60,popsize=8,polish=True)

## 3.5 — Probabilistic RBDO

For a specified: $\beta_T=2.5$, the design must satisfy
$$
\beta_j(\mathbf d)\ge\beta_T
$$

for every probabilistic mechanical limit state. The outer optimizer therefore searches for the least-cost design that satisfies an explicit reliability requirement.

In [ ]:
def wrb(x):
 b=wrel(x,WP[:1200])[1]; return wf(x)+8e4*np.sum(np.maximum(2.5-b,0)**2)
wp=differential_evolution(wrb,wb,seed=6,maxiter=70,popsize=8,polish=True)

## 3.6 — Fuzzy RBDO

At alpha=0 combine the upper load endpoint with the lower E and G endpoints to form the adverse full-support realization.

For $\widetilde x=(x_L,x_N,x_R)$,
$$
X_\alpha=[x_L+\alpha(x_N-x_L),\;x_R-\alpha(x_R-x_N)].
$$

The implementation uses **full-support protection** at $\alpha=0$. The adverse endpoint combination is propagated through the benchmark constraints and the design is optimized subject to feasibility at that realization.

In [ ]:
def wfg(x,a=0):
 P=acut((5280,6000,6720),a)[1]; E=acut((27.6e6,30e6,32.4e6),a)[0]; G=acut((11.04e6,12e6,12.96e6),a)[0]
 return wg(x,(P,E,G))
wfu=solve(wf,lambda x:wfg(x),wb,8,80)
display(pd.DataFrame([[n,*z,wf(z),np.min(wrel(z)[1])] for n,z in
 [("Deterministic",wd.x),("Robust",wr.x),("RBDO",wp.x),("Fuzzy RBDO",wfu.x)]],
 columns=["Method","h","l","t","b","Cost","min_beta"]))

,Method,h,l,t,b,Cost,min_beta
0,Deterministic,0.205729,3.470537,9.036668,0.205729,1.724863,-0.011029
1,Robust,0.210386,3.668628,9.158692,0.214869,1.852192,0.906257
2,RBDO,0.216107,3.941427,9.589767,0.218497,2.011962,2.619728
3,Fuzzy RBDO,0.216088,3.677251,9.253949,0.220340,1.923767,1.528453


## 3.7 — Benchmark comparison and interpretation

Compare the four resulting designs in terms of:

1. objective value and percentage increase from the deterministic baseline;
2. active/governing constraints;
3. reliability of the robust design under the probabilistic model;
4. objective penalty required to reach the RBDO target;
5. additional protection introduced by the full-support fuzzy formulation.

# 4. Benchmark III — Golinski Speed Reducer

### Engineering description
The speed reducer benchmark represents the sizing of a single-stage geared transmission. The design must minimize reducer weight while maintaining adequate gear strength, shaft stiffness/strength, and geometric compatibility. Its seven design variables make it useful for demonstrating how uncertainty analysis becomes more computationally demanding as design dimension and the number of active constraints increase.

The variables are
$$
\mathbf d=[b,m,z,L_1,L_2,d_1,d_2],
$$
where $b$ is gear-face width, $m$ is module, $z$ is the pinion tooth-count variable, $L_1,L_2$ are shaft bearing-span variables, and $d_1,d_2$ are shaft diameters.

### Mathematical formulation
The classical weight objective used here is
$$
\begin{aligned}
\min f(\mathbf d)=&\;0.7854bm^2(3.3333z^2+14.9334z-43.0934)\\
&-1.508b(d_1^2+d_2^2)+7.4777(d_1^3+d_2^3)\\
&+0.7854(L_1d_1^2+L_2d_2^2).
\end{aligned}
$$
The six principal mechanical constraints are
$$
g_1=\frac{27}{bm^2z}-1\le0,\qquad
 g_2=\frac{397.5}{bm^2z^2}-1\le0,
$$
$$
g_3=\frac{1.93L_1^3}{mzd_1^4}-1\le0,\qquad
 g_4=\frac{1.93L_2^3}{mzd_2^4}-1\le0,
$$
$$
g_5=\frac{\sqrt{(745L_1/(mz))^2+1.69\times10^7}}{110d_1^3}-1\le0,
$$
$$
g_6=\frac{\sqrt{(745L_2/(mz))^2+1.575\times10^8}}{85d_2^3}-1\le0.
$$
These are accompanied by geometric constraints
$$
g_7=\frac{mz}{40}-1\le0,
\quad g_8=\frac{5m}{b-1}-1\le0,
\quad g_9=\frac{b}{12m}-1\le0,
$$
$$
g_{10}=\frac{1.5d_1+1.9}{L_1}-1\le0,
\qquad
g_{11}=\frac{1.1d_2+1.9}{L_2}-1\le0.
$$
Bounds are
$$
b\in[2.6,3.6],\;m\in[0.7,0.8],\;z\in[17,28],
$$
$$
L_1,L_2\in[7.3,8.3],\;d_1\in[2.9,3.9],\;d_2\in[5.0,5.5].
$$

### Uncertainty model used in this notebook
The six mechanical constraints are multiplied by uncertain demand factors. Probabilistically,
$$
q_k\sim\mathcal N(1,0.04^2),
$$
while the fuzzy formulation uses
$$
\tilde q_k=(0.92,1.00,1.08).
$$
The geometric constraints remain deterministic. This is an instructional uncertainty extension.

## 4.1 — Model implementation

Encode the stated objective, constraints/limit states, and design bounds before introducing uncertainty.

In [ ]:
gb=[(2.6,3.6),(.7,.8),(17,28),(7.3,8.3),(7.3,8.3),(2.9,3.9),(5,5.5)]
def gf(x):
 b,m,z,L1,L2,d1,d2=x
 return .7854*b*m*m*(3.3333*z*z+14.9334*z-43.0934)-1.508*b*(d1*d1+d2*d2)+7.4777*(d1**3+d2**3)+.7854*(L1*d1*d1+L2*d2*d2)
def gg(x,q=np.ones(6)):
 b,m,z,L1,L2,d1,d2=x
 raw=np.array([27/(b*m*m*z),397.5/(b*m*m*z*z),1.93*L1**3/(m*z*d1**4),1.93*L2**3/(m*z*d2**4),
 np.sqrt((745*L1/(m*z))**2+1.69e7)/(110*d1**3),np.sqrt((745*L2/(m*z))**2+1.575e8)/(85*d2**3)])
 geom=np.array([m*z/40-1,5*m/(b-1)-1,b/(12*m)-1,(1.5*d1+1.9)/L1-1,(1.1*d2+1.9)/L2-1])
 return np.r_[q*raw-1,geom]

## 4.2 — Deterministic optimization

All uncertain quantities are fixed at their nominal values:

$$
\min_{\mathbf d} f(\mathbf d)
\quad\mathrm{s.t.}\quad
g_j(\mathbf d)\le0.
$$

The resulting design is the nominal baseline. The original constraints are evaluated after optimization to verify feasibility independently of the optimizer's status.

In [ ]:
gd=solve(gf,lambda x:gg(x),gb,9,170); print("DET weight",gf(gd.x)); display(check(gd.x,lambda x:gg(x)))

DET weight 3033.728658269128


,j,g,feasible
0,1,-0.099640,True
1,2,-0.220276,True
2,3,-0.497884,True
3,4,-0.904601,True
4,5,0.000001,False
5,6,0.000087,False
6,7,-0.702500,True
7,8,0.346154,False
8,9,-0.571429,True
9,10,-0.052140,True


## 4.3 — Uncertainty quantification of the deterministic optimum

Before redesigning the system, propagate the specified uncertainty through the deterministic optimum. For each safety margin $G_j=-g_j$,

$$
P_{f,j}\approx \frac{N(G_j<0)}{N},
\qquad
\beta_j=-\Phi^{-1}(P_{f,j}).
$$

This reveals whether the nominally feasible optimum remains reliable when uncertainty is introduced.

In [ ]:
rng=np.random.default_rng(33); QG=1+.04*rng.standard_normal((1800,6))
def grel(x,Q=QG):
 G=np.array([-gg(x,q)[:6] for q in Q]); pf=(G<0).mean(0)
 return pf,np.array([beta_pf(p,len(Q)) for p in pf])

## 4.4 — Robust optimization

**Purpose:** reduce sensitivity to uncertainty rather than enforce a prescribed failure probability.

Use the maximum of the six uncertain mechanical constraints as the response. Penalize its positive mean and variability; the five geometric constraints remain deterministic.

A generic form is

$$
J_R(\mathbf d)=f(\mathbf d)+\lambda\,\sigma_D(\mathbf d)
+
\rho\,[\max(\mu_D(\mathbf d),0)]^2,
$$

where $D$ is an uncertainty-dependent demand/violation measure. The mean term discourages adverse average behavior and the standard-deviation term discourages variability.

In [ ]:
def grob(x):
 v=np.array([np.max(gg(x,q)[:6]) for q in QG[:350]])
 return gf(x)+2*np.std(v)+5e4*max(v.mean(),0)**2+1e7*np.sum(np.maximum(gg(x)[6:],0)**2)
gr=differential_evolution(grob,gb,seed=11,maxiter=45,popsize=7,polish=True)

## 4.5 — Probabilistic RBDO

For a specified $\beta_T=2.0$, the design must satisfy
$$
\beta_j(\mathbf d)\ge\beta_T
$$

for every probabilistic mechanical limit state. The outer optimizer therefore searches for the least-cost design that satisfies an explicit reliability requirement.

In [ ]:
def grb(x):
 b=grel(x,QG[:800])[1]; return gf(x)+2e4*np.sum(np.maximum(2-b,0)**2)+1e7*np.sum(np.maximum(gg(x)[6:],0)**2)
gp=differential_evolution(grb,gb,seed=10,maxiter=55,popsize=7,polish=True)

## 4.6 — Fuzzy RBDO

At alpha=0 use q=1.08 for the six uncertain mechanical demands; geometric constraints remain deterministic.

For $\widetilde x=(x_L,x_N,x_R)$,
$$
X_\alpha=[x_L+\alpha(x_N-x_L),\;x_R-\alpha(x_R-x_N)].
$$

The implementation uses **full-support protection** at $\alpha=0$. The adverse endpoint combination is propagated through the benchmark constraints and the design is optimized subject to feasibility at that realization.

In [ ]:
def gfg(x,a=0): return gg(x,np.array([acut((.92,1,1.08),a)[1]]*6))
gfu=solve(gf,lambda x:gfg(x),gb,12,65)
display(pd.DataFrame([[n,gf(z),np.min(grel(z)[1])] for n,z in
 [("Deterministic",gd.x),("Robust",gr.x),("RBDO",gp.x),("Fuzzy RBDO",gfu.x)]],
 columns=["Method","Weight","min_beta"]))

,Method,Weight,min_beta
0,Deterministic,3033.728658,0.004178
1,Robust,3030.833972,-0.301106
2,RBDO,3409.960627,1.849093
3,Fuzzy RBDO,3166.569581,1.941297


## 4.7 — Benchmark comparison and interpretation

Compare the four resulting designs in terms of:

1. objective value and percentage increase from the deterministic baseline;
2. active/governing constraints;
3. reliability of the robust design under the probabilistic model;
4. objective penalty required to reach the RBDO target;
5. additional protection introduced by the full-support fuzzy formulation.

# 5. Benchmark IV — Cantilever Beam RBDO

### Engineering description
A rectangular cantilever beam is subjected to two uncertain transverse load components, $X$ and $Y$. The designer chooses the cross-section dimensions $w$ and $t$. The goal is to minimize cross-sectional area while preventing **stress failure** and excessive **tip displacement**. Unlike the first three examples, this problem is introduced directly as a reliability-based benchmark, making it especially useful for illustrating probabilistic limit states.

### Mathematical formulation
$$
\mathbf d=[w,t],\qquad \min C(w,t)=wt.
$$
Using the reliability convention $G>0$ safe and $G<0$ failure, the stress margin is
$$
G_1=Y_s-\left(\frac{600Y}{wt^2}+\frac{600X}{w^2t}\right),
$$
and the displacement margin is
$$
G_2=2.5-\frac{4L^3}{Ewt}
\sqrt{\left(\frac{Y}{t^2}\right)^2+\left(\frac{X}{w^2}\right)^2}.
$$
Thus the probabilistic RBDO problem is
$$
\min_{w,t}\;wt
$$
subject to
$$
\beta_1(w,t)\ge\beta_T,\qquad
\beta_2(w,t)\ge\beta_T,
$$
with
$$
\beta_j=-\Phi^{-1}\!\left(P[G_j<0]\right).
$$
This notebook uses $L=100$, $w,t\in[0.5,10]$, and $\beta_T=3.012$.

For the instructional probabilistic implementation,
$$
X\sim N(500,50^2),\quad Y\sim N(1000,100^2),
$$
$$
Y_s\sim N(40000,2000^2),\quad
E\sim N(29\times10^6,(1.45\times10^6)^2).
$$

### Fuzzy extension
The corresponding triangular fuzzy variables are
$$
\tilde X=(300,500,700),\quad
\tilde Y=(800,1000,1200),
$$
$$
\tilde Y_s=(36000,40000,44000),\quad
\tilde E=(26.1,29,31.9)\times10^6.
$$
At each $\alpha$-cut, the uncertain inputs are propagated through $G_1,G_2$. A conservative full-support fuzzy design requires the lower safety-margin bound to satisfy
$$
G_{j,L}(0)\ge0,\qquad j=1,2.
$$

## 5.1 — Model implementation

Encode the stated objective, constraints/limit states, and design bounds before introducing uncertainty.

In [ ]:
bb=[(.5,10),(.5,10)]; L=100.
def bf(x): return x[0]*x[1]
def bG(x,p=(500.,1000.,40000.,29e6)):
 w,t=x; X,Y,Ys,E=p
 G1=Ys-(600*Y/(w*t*t)+600*X/(w*w*t))
 disp=4*L**3/(E*w*t)*np.sqrt((Y/t**2)**2+(X/w**2)**2)
 return np.array([G1,2.5-disp])

## 5.2 — Deterministic optimization

All uncertain quantities are fixed at their nominal values:

$$
\min_{\mathbf d} f(\mathbf d)
\quad\mathrm{s.t.}\quad
g_j(\mathbf d)\le0.
$$

The resulting design is the nominal baseline. The original constraints are evaluated after optimization to verify feasibility independently of the optimizer's status.

In [ ]:
bd=solve(bf,lambda x:-bG(x),bb,13,130); print("DET",bd.x,bf(bd.x),bG(bd.x))

DET [2.04700531 3.74602521] 7.668133516698588 [1.92225735e-03 1.61271584e-06]


## 5.3 — Uncertainty quantification of the deterministic optimum

Before redesigning the system, propagate the specified uncertainty through the deterministic optimum. For each safety margin $G_j=-g_j$,

$$
P_{f,j}\approx \frac{N(G_j<0)}{N},
\qquad
\beta_j=-\Phi^{-1}(P_{f,j}).
$$

This reveals whether the nominally feasible optimum remains reliable when uncertainty is introduced.

In [ ]:
rng=np.random.default_rng(44); n=5000
BP=np.c_[rng.normal(500,50,n),rng.normal(1000,100,n),rng.normal(40000,2000,n),rng.normal(29e6,1.45e6,n)]
def brel(x,PP=BP):
 G=np.array([bG(x,p) for p in PP]); pf=(G<0).mean(0)
 return pf,np.array([beta_pf(p,len(PP)) for p in pf]),G.mean(0),G.std(0)

## 5.4 — Robust optimization

**Purpose:** reduce sensitivity to uncertainty rather than enforce a prescribed failure probability.

Use the two physical safety margins. Their means and standard deviations define a standardized robustness buffer. This is a variability-aware criterion, not the later probability-based reliability constraint.

A generic form is

$$
J_R(\mathbf d)=f(\mathbf d)+\lambda\,\sigma_D(\mathbf d)
+
\rho\,[\max(\mu_D(\mathbf d),0)]^2,
$$

where $D$ is an uncertainty-dependent demand/violation measure. The mean term discourages adverse average behavior and the standard-deviation term discourages variability.

In [ ]:
def brob(x):
 _,_,mu,sd=brel(x,BP[:1200]); r=np.min(mu/(sd+1e-12))
 return bf(x)+2e4*max(1.5-r,0)**2
br=differential_evolution(brob,bb,seed=15,maxiter=60,popsize=8,polish=True)

## 5.5 — Probabilistic RBDO

The problem specifies $\beta_T=3.012$, the design must satisfy
$$
\beta_j(\mathbf d)\ge\beta_T
$$

for every probabilistic mechanical limit state. The outer optimizer therefore searches for the least-cost design that satisfies an explicit reliability requirement.

In [ ]:
def brb(x):
 b=brel(x,BP[:2500])[1]; return bf(x)+2e4*np.sum(np.maximum(3.012-b,0)**2)
bp=differential_evolution(brb,bb,seed=14,maxiter=75,popsize=9,polish=True)

## 5.6 — Fuzzy RBDO

At alpha=0 combine high X and Y with low yield strength and low elastic modulus. Both lower safety-margin bounds must remain nonnegative.

For $\widetilde x=(x_L,x_N,x_R)$,
$$
X_\alpha=[x_L+\alpha(x_N-x_L),\;x_R-\alpha(x_R-x_N)].
$$

The implementation uses **full-support protection** at $\alpha=0$. The adverse endpoint combination is propagated through the benchmark constraints and the design is optimized subject to feasibility at that realization.

In [ ]:
BF=[(300,500,700),(800,1000,1200),(36000,40000,44000),(26.1e6,29e6,31.9e6)]
def bfg(x,a=0):
 X=acut(BF[0],a)[1]; Y=acut(BF[1],a)[1]; Ys=acut(BF[2],a)[0]; E=acut(BF[3],a)[0]
 return bG(x,(X,Y,Ys,E))
bfu=solve(bf,lambda x:-bfg(x),bb,16,85)
display(pd.DataFrame([[n,*z,bf(z),*brel(z)[1]] for n,z in
 [("Deterministic",bd.x),("Robust",br.x),("RBDO",bp.x),("Fuzzy RBDO",bfu.x)]],
 columns=["Method","w","t","Area","beta_stress","beta_disp"]))

,Method,w,t,Area,beta_stress,beta_disp
0,Deterministic,2.047005,3.746025,7.668134,0.007520,-0.023564
1,Robust,2.079057,4.015414,8.348274,1.496974,1.495439
2,RBDO,2.278521,3.924703,8.942519,2.988882,3.719016
3,Fuzzy RBDO,2.389572,4.089708,9.772650,3.719016,3.719016


## 5.7 — Benchmark comparison and interpretation

Compare the four resulting designs in terms of:

1. objective value and percentage increase from the deterministic baseline;
2. active/governing constraints;
3. reliability of the robust design under the probabilistic model;
4. objective penalty required to reach the RBDO target;
5. additional protection introduced by the full-support fuzzy formulation.


# 6. Final cross-benchmark synthesis

One final comparison of the four formulations. The percentage objective penalty is

$$
\%\Delta f=
100\left(\frac{f_{\mathrm{method}}}{f_{\mathrm{det}}}-1\right).
$$

A positive value is the cost paid, in the benchmark objective, for uncertainty protection.

In [ ]:
summary=pd.DataFrame([
["Spring","Deterministic",sf(sd.x)],["Spring","Robust",sf(sr.x)],["Spring","RBDO",sf(sp.x)],["Spring","Fuzzy RBDO",sf(sfu.x)],
["Welded beam","Deterministic",wf(wd.x)],["Welded beam","Robust",wf(wr.x)],["Welded beam","RBDO",wf(wp.x)],["Welded beam","Fuzzy RBDO",wf(wfu.x)],
["Speed reducer","Deterministic",gf(gd.x)],["Speed reducer","Robust",gf(gr.x)],["Speed reducer","RBDO",gf(gp.x)],["Speed reducer","Fuzzy RBDO",gf(gfu.x)],
["Cantilever","Deterministic",bf(bd.x)],["Cantilever","Robust",bf(br.x)],["Cantilever","RBDO",bf(bp.x)],["Cantilever","Fuzzy RBDO",bf(bfu.x)]
],columns=["Benchmark","Method","Objective"])
summary["Penalty_vs_deterministic_%"]=summary.groupby("Benchmark")["Objective"].transform(lambda s:100*(s/s.iloc[0]-1))
display(summary.round(4))

,Benchmark,Method,Objective,Penalty_vs_deterministic_%
0,Spring,Deterministic,0.0127,0.0000
1,Spring,Robust,0.0138,8.4708
2,Spring,RBDO,0.0180,41.3783
3,Spring,Fuzzy RBDO,0.0141,11.0607
4,Welded beam,Deterministic,1.7249,0.0000
5,Welded beam,Robust,1.8522,7.3820
6,Welded beam,RBDO,2.0120,16.6447
7,Welded beam,Fuzzy RBDO,1.9238,11.5316
8,Speed reducer,Deterministic,3033.7287,0.0000
9,Speed reducer,Robust,3030.8340,-0.0954


## Final conclusions

- The **deterministic optimum** is a necessary baseline but contains no explicit protection against uncertainty.
- **Robust optimization** changes the design because variability itself is undesirable; its reliability must still be measured afterward.
- **Probabilistic RBDO** makes reliability an explicit constraint through $P_f$ or $\beta$.
- **Fuzzy RBDO** is appropriate when uncertainty is represented as imprecise/possibilistic information rather than a defensible probability distribution.
- The design penalty associated with RBDO or fuzzy RBDO should be interpreted as the engineering cost of the selected uncertainty-protection requirement.
- Final designs should be independently re-evaluated with a larger Monte Carlo sample or higher-fidelity model before implementation.